# Problem Statement 42 — Indian Sign Language Recognition

This notebook uses the shared project pipeline to inspect the ISLAN dataset, train a CNN, and evaluate it on a held-out test split. The dataset must be arranged in one folder per sign class. Motion signs **J** and **Z** are excluded. Horizontal flipping is intentionally disabled because it may change handedness or sign meaning.

Run all cells after installing `requirements.txt` and placing the dataset in `data/ISLAN` (or update `DATASET_PATH`).

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

from problem_statement_42_solution import (
    DEFAULT_DATASET_PATH,
    collect_dataset,
    save_dataset_overview,
)

DATASET_PATH = DEFAULT_DATASET_PATH
OUTPUT_DIR = Path(__file__).resolve().parent / "outputs" if "__file__" in globals() else Path.cwd() / "outputs"
SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 40

print(f"Dataset: {DATASET_PATH}")
print(f"Artifacts: {OUTPUT_DIR}")

In [ ]:
image_paths, labels, class_names = collect_dataset(DATASET_PATH)
counts = Counter(labels.tolist())
for class_id, class_name in enumerate(class_names):
    print(f"{class_name}: {counts[class_id]} images")

save_dataset_overview(image_paths, labels, class_names, OUTPUT_DIR)
display(plt.imread(OUTPUT_DIR / "class_counts.png"))
display(plt.imread(OUTPUT_DIR / "sample_images.png"))

## Training and evaluation

The pipeline makes stratified 60/20/20 train/validation/test splits using the fixed seed. Validation data is used for early stopping and checkpoint selection; the held-out test set is used only for final metrics.

In [ ]:
from problem_statement_42_solution import build_and_train_model

test_results = build_and_train_model(
    dataset_path=DATASET_PATH,
    output_dir=OUTPUT_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    seed=SEED,
)
test_results

In [ ]:
from IPython.display import Image, display

for artifact in [
    "training_curves.png",
    "confusion_matrix.png",
    "misclassified_examples.png",
    ]:
    artifact_path = OUTPUT_DIR / artifact
    if artifact_path.exists():
        print(artifact)
        display(Image(filename=str(artifact_path)))

print(f"Detailed test metrics: {OUTPUT_DIR / 'test_metrics.json'}")